In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rm1000/brain-tumor-mri-scans")

print("Path to dataset files:", path)

Path to dataset files: /root/.cache/kagglehub/datasets/rm1000/brain-tumor-mri-scans/versions/1


In [2]:
import os
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16
import datetime
import math
import matplotlib.pyplot as plt

os.listdir(path)

['glioma', 'pituitary', 'meningioma', 'healthy']

In [4]:
BATCH_SIZE = 32
BUFFER_SIZE = 1000
IMG_SIZE = (224, 224)
SEED = 42
EPOCHS = 10

In [5]:
train_ds = tf.keras.utils.image_dataset_from_directory(
  path,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=IMG_SIZE,
  batch_size=BATCH_SIZE)


val_ds = tf.keras.utils.image_dataset_from_directory(
  path,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=IMG_SIZE,
  batch_size=BATCH_SIZE)

Found 7023 files belonging to 4 classes.
Using 5619 files for training.
Found 7023 files belonging to 4 classes.
Using 1404 files for validation.


In [6]:
class_names = train_ds.class_names
print(class_names)
num_classes = len(class_names)

['glioma', 'healthy', 'meningioma', 'pituitary']


In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(class_names[labels[i]])
    plt.axis("off")

In [8]:
for image_batch, labels_batch in train_ds:
  print(image_batch.shape)
  print(labels_batch.shape)
  break

(32, 224, 224, 3)
(32,)


In [9]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(BUFFER_SIZE).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

data_augmentation = tf.keras.Sequential(
  [
    layers.RandomFlip("horizontal", input_shape=(224, 224,3)),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
  ]
)

/usr/local/lib/python3.10/dist-packages/keras/src/layers/preprocessing/tf_data_layer.py:19: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
plt.figure(figsize=(10, 10))
for images, _ in train_ds.take(1):
  for i in range(9):
    augmented_images = data_augmentation(images)
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(augmented_images[0].numpy().astype("uint8"))
    plt.axis("off")

In [16]:
def make_model():
    # Modelo base VGG16 pré-treinado no ImageNet
    base_model = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False  # Congela as camadas do VGG16

    # Define o modelo completo com o VGG16 como base
    model = tf.keras.Sequential([
        data_augmentation,  # Aumento de dados
        layers.Rescaling(1./255),  # Normalização
        base_model,  # Modelo pré-treinado
        layers.Flatten(),  # Flatten após VGG16
        layers.Dropout(0.2),  # Regularização
        layers.Dense(128, activation='relu'),  # Camada densa para aprendizado
        layers.Dense(num_classes, activation="softmax")  # Saída final com softmax
    ])

    # Compila o modelo
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
                  metrics=['accuracy'])

    return model

In [17]:
def step_decay(epoch):
	initial_lr = 0.01
	drop = 0.5
	epochs_drop = 1
	lr = initial_lr * math.pow(drop, math.floor((1+epoch)/epochs_drop))
	return lr

lrate = tf.keras.callbacks.LearningRateScheduler(step_decay)

In [ ]:
model = make_model()

history = model.fit(train_ds,
                    validation_data=val_ds,
                    epochs=EPOCHS,
                    callbacks=[lrate])

Epoch 1/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 4030s 23s/step - accuracy: 0.6453 - loss: 5.0033 - val_accuracy: 0.8868 - val_loss: 0.3199 - learning_rate: 0.0050
Epoch 2/10
 75/176 ━━━━━━━━━━━━━━━━━━━━ 30:43 18s/step - accuracy: 0.8554 - loss: 0.3924